In [4]:
from langchain_openai import ChatOpenAI
from langchain.agents.format_scratchpad.openai_tools import (
    format_to_openai_tool_messages,
)
from langchain.agents.output_parsers.openai_tools import OpenAIToolsAgentOutputParser
from langchain.agents import AgentExecutor
from langchain_core.messages import AIMessage, HumanMessage
from tools.material_extraction import (
    get_material_ids,
    MPSurfacePropertiesTool,
    MPPhononTool,
    MPElasticityTool,
    MPThermoTool,
    MPDielectricTool,
    MPPiezoelectricTool,
    MPMagnetismTool,
    MPSynthesisTool,
    MPElectronicStructureTool,
    MPElectronicBandStructureTool,
    MPElectronicDensityOfStatesTool,
    MPOxidationStatesTool,
    MPBondsTool,
    MPAbsorptionTool,
    extract_materials_matweb
)
from tools.material_extraction import OPENAI_API

In [5]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, api_key=OPENAI_API)

tools = [
    get_material_ids,
    MPSurfacePropertiesTool(),
    MPPhononTool(),
    MPElasticityTool(),
    MPThermoTool(),
    MPDielectricTool(),
    MPPiezoelectricTool(),
    MPMagnetismTool(),
    MPSynthesisTool(),
    MPElectronicStructureTool(),
    MPElectronicBandStructureTool(),
    MPElectronicDensityOfStatesTool(),
    MPOxidationStatesTool(),
    MPBondsTool(),
    MPAbsorptionTool(),
    extract_materials_matweb
]
MEMORY_KEY = "chat_history"
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate, MessagesPlaceholder

system_message = SystemMessagePromptTemplate.from_template(
    """
    You are a powerful agent for performing data extraction from the Materials Project and matweb.com.
    Your task is to determine the type of problem in the user's query before taking action. 
    First, classify the problem into one of the following types:
    1. Finding materials based on given physical properties (e.g., Young's modulus, tensile strength).
    2. Getting properties of a specific material (e.g., getting the properties of mp-149).
    
    Based on your classification, proceed with the appropriate action:
    - If it's Type 1 (Finding materials based on properties), explain the tool you are using (e.g., searching for materials based on physical properties), what parameters you are using, and why you chose those parameters. Then, execute the appropriate action to search for materials.
    - If it's Type 2 (Getting properties of a specific material), first find the material_id(s) for the material, then explain the tool you are using (e.g., fetching material properties), the parameters, and the reasoning behind those choices. Then, execute the appropriate action to retrieve the properties of the specific material.
    
    DO NOT MAKE YOUR OWN DATA UP. If Materials Project does not return any results, state that and ask for more information.
    After running the extraction from materials project, if the problem is type 1, extract from matweb as well if the property is
     relevant to the agent
    """
)

# human message template
human_message = HumanMessagePromptTemplate.from_template(
    """
    Given the user's query: '{input}', first classify the problem as either '1' for finding materials
 based on physical properties or '2' for getting material properties. If it is '2', then state the chemical
 formula of the  specified materials and its contituent elements seperately. Use the chemical formula to get material_id(s)
 first then proceed. If it is type '1' use the appropiate tool.
    """
)

prompt = ChatPromptTemplate.from_messages(
    [
        system_message,
        MessagesPlaceholder(variable_name=MEMORY_KEY),
        human_message,
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

llm_with_tools = llm.bind_tools(tools)

agent = (
    {
        "input": lambda x: x["input"],
        "agent_scratchpad": lambda x: format_to_openai_tool_messages(
            x["intermediate_steps"]
        ),
        "chat_history": lambda x: x["chat_history"],
    }
    | prompt
    | llm_with_tools
    | OpenAIToolsAgentOutputParser()
)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [6]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
chat_history = []

input1 = """
Provide synthesis details for perovskite oxides used in solar cells.
"""
result = agent_executor.invoke({"input": input1, "chat_history": chat_history})
chat_history.extend(
    [
        HumanMessage(content=input1),
        AIMessage(content=result["output"]),
    ]
)



> Entering new AgentExecutor chain...

Invoking: `MaterialsProjectSynthesis` with `{'keywords': ['perovskite oxides'], 'num_chunks': 1, 'chunk_size': 10}`
responded: This query falls under Type 1: Finding materials based on physical properties. 
I will use the Materials Project Synthesis tool to search for synthesis details for perovskite oxides used in solar cells. 
I will search for materials based on the keyword "perovskite oxides" in the synthesis details.

The study investigated solid-state synthesis of (Bi1-xSrx)CrO3 compositions with 0.2 ≤ x ≤ 0.8 using a reaction involving Bi2O3, Cr2O3, O2, SrCO3 to form (Bi1-xSrx)CrO3 and CO2. X-ray analysis revealed formation of a single perovskite phase only at x = 0.5. The study highlights the synthesis and characterization of various (Bi1-xSrx)CrO3 compositions, showcasing the perovskite-type phase formation. The results shed light on the potential for tuning properties of perovskite materials through compositional variations, with x imp

In [7]:
chat_history = []

input1 = """
What materials have a Shear Modulus between 300 and 350GPa? List their properties.
"""
result = agent_executor.invoke({"input": input1, "chat_history": chat_history})
chat_history.extend(
    [
        HumanMessage(content=input1),
        AIMessage(content=result["output"]),
    ]
)



> Entering new AgentExecutor chain...

Invoking: `MaterialsProjectElasticity` with `{'g_voigt': [300, 350], 'chunk_size': 10}`
responded: This problem falls under Type 1: Finding materials based on physical properties. 
I will use the Materials Project API to search for materials with a Shear Modulus between 300 and 350 GPa. Let me perform the search.

The results pertain to a material with the chemical system Be-C-N, having a composition of BeCN2. The material exhibits a tetragonal crystal system with a volume of 49.43 Angstrom³ and a density of 3.29 g/cm³. The elasticity property is highlighted, with an elastic tensor detailed for the material. The material's bulk modulus is reported at 314.97 GPa, while the shear modulus is 292.03 GPa. The material displays anisotropic behavior with a universal anisotropy of 0.43 and homogeneous Poisson ratio of 0.146. The Debye temperature is recorded at 1677.1 K. The material has a sound velocity of 540.37 m/s and thermal conductivity values of 

In [8]:
chat_history = []

input1 = """
Retrieve bond length and coordination information for TiO2. List your results."
"""
result = agent_executor.invoke({"input": input1, "chat_history": chat_history})
chat_history.extend(
    [
        HumanMessage(content=input1),
        AIMessage(content=result["output"]),
    ]
)



> Entering new AgentExecutor chain...

Invoking: `get_material_ids` with `{'compound': 'TiO2'}`
responded: This query falls under Type 2, which involves getting properties of a specific material. 
The specified material is TiO2, which has the chemical formula of TiO2. 
I will first retrieve the material_id(s) for TiO2 using the get_material_ids tool before proceeding to extract the bond length and coordination information for TiO2.

[MPID(mp-390), MPID(mp-430), MPID(mp-1439), MPID(mp-1840), MPID(mp-2657), MPID(mp-9173), MPID(mp-25262), MPID(mp-25433), MPID(mp-34688), MPID(mp-553432), MPID(mp-554278), MPID(mp-572822), MPID(mp-636827), MPID(mp-655656), MPID(mp-754672), MPID(mp-754769), MPID(mp-756587), MPID(mp-756655), MPID(mp-766454), MPID(mp-775938), MPID(mp-777129), MPID(mp-849533), MPID(mp-1008677), MPID(mp-1041565), MPID(mp-1044853), MPID(mp-1079207), MPID(mp-1101022), MPID(mp-1102255), MPID(mp-1102591), MPID(mp-1190753), MPID(mp-1201727), MPID(mp-1245098), MPID(mp-1245134), MPID(

In [9]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
chat_history = []

input1 = """
Find materials that have 1 to 3 unique magnetic sites with ferromagnetic ordering. The total magnetization should range from 1 to 5 µB.
"""
result = agent_executor.invoke({"input": input1, "chat_history": chat_history})
chat_history.extend(
    [
        HumanMessage(content=input1),
        AIMessage(content=result["output"]),
    ]
)



> Entering new AgentExecutor chain...

Invoking: `MaterialsProjectMagnetism` with `{'num_unique_magnetic_sites': [1, 3], 'ordering': 'FM', 'total_magnetization': [1, 5], 'num_chunks': 5}`
responded: This problem falls under Type 1: Finding materials based on physical properties. 
I will use the Materials Project API to search for materials that meet the specified criteria of having 1 to 3 unique magnetic sites with ferromagnetic ordering and a total magnetization ranging from 1 to 5 µB. 

Let's proceed with searching for materials based on these physical properties.



c:\Users\619ar\anaconda3\envs\mat-proj\Lib\site-packages\mp_api\client\core\client.py:829: UserWarning: Use the 'fields' argument to select only fields of interest to speed up data retrieval for large queries. Choose from: ['builder_meta', 'nsites', 'elements', 'nelements', 'composition', 'composition_reduced', 'formula_pretty', 'formula_anonymous', 'chemsys', 'volume', 'density', 'density_atomic', 'symmetry', 'property_name', 'material_id', 'deprecated', 'deprecation_reasons', 'last_updated', 'origins', 'warnings', 'ordering', 'is_magnetic', 'exchange_symmetry', 'num_magnetic_sites', 'num_unique_magnetic_sites', 'types_of_magnetic_species', 'magmoms', 'total_magnetization', 'total_magnetization_normalized_vol', 'total_magnetization_normalized_formula_units']
  warnings.warn(


The results show a compound with a formula Rb3TiF6, consisting of the elements Rubidium (Rb), Titanium (Ti), and Fluorine (F). The compound exhibits cubic symmetry with a crystal system of Fm-3m. It is magnetic with a total magnetization of 1.0, primarily attributed to the presence of magnetic species Titanium (Ti). The compound has a volume of 191.88 Å^3 and a density of 3.62 g/cm^3. The compound's magnetism is ordered in the ferromagnetic (FM) state and displays exchange symmetry of 139. These findings provide valuable insights into the magnetic properties of this material, highlighting potential applications in magnetic technologies. The results describe a material with 42 sites and 5 elements (Ce, Cu, Nd, O, Sr) in its composition. The compound has a formula of Sr8Ce5Nd3(CuO12)2, belongs to the Ce-Cu-Nd-O-Sr chemical system, and crystallizes in a monoclinic structure with a density of 5.32 g/cm³. It exhibits magnetic properties with a total magnetization of 1.0 and contains Cu as t